# ROGII Wellbore Geology - Dense (MLP) Inference

Load the trained Multi-Layer Perceptron (Dense) model, predict TVT for test wells.

**Runtime**: Kaggle GPU - **Author**: Samir Attrah

In [ ]:
# Cell 1: Environment & Imports
import glob
import os
import pickle
import warnings

os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import jax

jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import keras
import numpy as np
import polars as pl

warnings.filterwarnings("ignore")
print(f"Keras: {keras.__version__}, Backend: {keras.backend.backend()}")


In [7]:
# Cell 2: Auto-detect dataset path
def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/home/samer/Documents/competitions/ROGII/dataset",
    ]
    for path in candidates:
        if os.path.isdir(path):
            if "test" in os.listdir(path):
                return path
    raise FileNotFoundError("Dataset not found.")


DATA_DIR = find_data_dir()


In [ ]:
# Cell 3: Auto-detect model artifact
def find_model_dir():
    candidates = [
        "/home/samer/Documents/competitions/ROGII/outputs",
        "/kaggle/working",
        "/kaggle/input/models/samerattrah/rogii-21-05/keras/default/17",
    ]
    for path in candidates:
        model_path = os.path.join(path, "optimized_dense_model.keras")
        scaler_path = os.path.join(path, "dense_opt_scaler_params.pkl")
        if (
            os.path.isdir(path)
            and os.path.exists(model_path)
            and os.path.exists(scaler_path)
        ):
            return path
    raise FileNotFoundError(
        "Could not find optimized dense model and scaler artifacts."
    )


MODEL_DIR = find_model_dir()
print(f"Using model artifacts from: {MODEL_DIR}")


In [9]:
# Cell 4: Configuration
OUT_DIR = (
    "/kaggle/working"
    if os.path.isdir("/kaggle")
    else "/home/samer/Documents/competitions/ROGII/outputs"
)
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = {
    "data_dir": DATA_DIR,
    "model_path": os.path.join(MODEL_DIR, "optimized_dense_model.keras"),
    "scaler_path": os.path.join(MODEL_DIR, "dense_opt_scaler_params.pkl"),
    "submission_path": os.path.join(OUT_DIR, "submission.csv"),
}
FEATURE_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]


In [10]:
# Cell 5: Load model & scaler
print("Loading model...")
model = keras.saving.load_model(CONFIG["model_path"])
print("Loading scaler...")
with open(CONFIG["scaler_path"], "rb") as f:
    scaler = pickle.load(f)


Loading model...
Loading scaler...


In [ ]:
# Cell 6: Prediction Logic
def preprocess(df, feature_cols):
    """Preprocess one well with inference-safe handling of missing features."""
    for col in feature_cols:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col)
                .interpolate()
                .fill_null(strategy="forward")
                .fill_null(strategy="backward")
                .fill_null(0.0)
            )
    return df


def get_submission_index(sample_sub):
    """Adds well_id and zero-based row_idx parsed from sample_submission ids."""
    return sample_sub.with_columns(
        [
            pl.col("id").str.extract(r"^(.+)_(\d+)$", 1).alias("well_id"),
            pl.col("id")
            .str.extract(r"^(.+)_(\d+)$", 2)
            .cast(pl.Int64)
            .alias("row_idx"),
        ]
    )


def normalize_features(df, scaler):
    """Applies the exact scaler saved by the dense optimization notebook."""
    feature_cols = scaler.get("feature_cols", FEATURE_COLS)
    raw_feats = df.select(feature_cols).to_numpy()
    feats = jnp.array(raw_feats, dtype=jnp.float64)
    mean = jnp.array(scaler["feat_mean"], dtype=jnp.float64)
    std = jnp.array(scaler["feat_std"], dtype=jnp.float64)
    std = jnp.where(std == 0, 1.0, std)
    return (feats - mean) / std


def denormalize_target(yn, scaler):
    """Converts normalized model outputs back to raw TVT units."""
    target_mean = jnp.array(scaler["target_mean"], dtype=jnp.float64)
    target_std = jnp.array(scaler["target_std"], dtype=jnp.float64)
    return (jnp.array(yn, dtype=jnp.float64) * target_std) + target_mean


def smooth_prediction_zone(preds, row_idxs, window=11, anchor_weight=0.15):
    """Smooths only submitted rows and softly anchors the first row to continuity.

    This reduces row-to-row noise from a pointwise dense model without changing known
    non-submission rows. The first submitted prediction is blended slightly toward the
    previous prediction to avoid a sharp jump at the TVT_input cutoff.
    """
    if len(row_idxs) == 0:
        return preds

    preds = preds.copy()
    zone = np.asarray(row_idxs, dtype=np.int64)
    zone_vals = preds[zone]

    if len(zone_vals) >= window:
        kernel = np.ones(window, dtype=np.float64) / window
        pad_left = window // 2
        pad_right = window - 1 - pad_left
        padded = np.pad(zone_vals, (pad_left, pad_right), mode="edge")
        zone_vals = np.convolve(padded, kernel, mode="valid")

    first_idx = int(zone[0])
    if first_idx > 0:
        zone_vals[0] = (1.0 - anchor_weight) * zone_vals[0] + anchor_weight * preds[
            first_idx - 1
        ]

    preds[zone] = zone_vals
    return preds


def predict_well(model, df, scaler, row_idxs=None, postprocess=True):
    """Inference for one well using the optimized Dense model."""
    feature_cols = scaler.get("feature_cols", FEATURE_COLS)
    df = preprocess(df, feature_cols)
    feats_n = normalize_features(df, scaler)

    X = np.array(feats_n, dtype=np.float32)
    yn = model.predict(X, batch_size=2048, verbose=0).ravel()
    yp = np.array(denormalize_target(yn, scaler), dtype=np.float64)

    if postprocess and row_idxs is not None:
        yp = smooth_prediction_zone(yp, row_idxs)

    return yp


In [ ]:
# Cell 7: Run Inference
sample_sub = get_submission_index(
    pl.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
)
submission_windows = {
    row["well_id"]: list(range(row["min_idx"], row["max_idx"] + 1))
    for row in sample_sub.group_by("well_id")
    .agg(pl.min("row_idx").alias("min_idx"), pl.max("row_idx").alias("max_idx"))
    .iter_rows(named=True)
}
test_ids = sorted(submission_windows)

all_preds = {}
for wid in test_ids:
    print(f"Predicting {wid}...")
    well_path = os.path.join(DATA_DIR, "test", f"{wid}__horizontal_well.csv")
    df = pl.read_csv(well_path, infer_schema_length=10000)
    row_idxs = submission_windows[wid]
    all_preds[wid] = predict_well(model, df, scaler, row_idxs=row_idxs)

rows = []
missing = 0
for r in sample_sub.iter_rows(named=True):
    arr = all_preds.get(r["well_id"])
    if arr is None or r["row_idx"] >= len(arr):
        val = 0.0
        missing += 1
    else:
        val = float(arr[r["row_idx"]])
    rows.append({"id": r["id"], "tvt": val})

if missing:
    print(
        f"Warning: {missing} submission rows were missing predictions and were filled with 0.0"
    )

submission = pl.DataFrame(rows)
submission.write_csv(CONFIG["submission_path"])
print(f"Submission saved to {CONFIG['submission_path']}")
print(
    submission.select(
        pl.col("tvt").min().alias("min"),
        pl.col("tvt").max().alias("max"),
        pl.col("tvt").mean().alias("mean"),
    )
)


In [ ]:
# Cell 8: Visualise Predictions
import matplotlib.pyplot as plt

ANALYTICS_DIR = "../analytics"
os.makedirs(ANALYTICS_DIR, exist_ok=True)


def plot_predictions(all_preds, test_ids, submission_windows):
    n_wells = len(test_ids)
    if n_wells == 0:
        return

    fig, axes = plt.subplots(n_wells, 1, figsize=(15, 5 * n_wells), sharex=False)
    if n_wells == 1:
        axes = [axes]

    for i, wid in enumerate(test_ids):
        preds = all_preds[wid]
        axes[i].plot(preds, label="Predicted TVT", color="blue", lw=1)
        rows = submission_windows.get(wid, [])
        if rows:
            axes[i].axvspan(
                rows[0], rows[-1], color="orange", alpha=0.15, label="Submission rows"
            )
        axes[i].set_title(f"Well {wid} - Predicted TVT (Dense)")
        axes[i].set_ylabel("TVT")
        axes[i].legend()
        axes[i].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYTICS_DIR, "test_predictions_plots_dense.png"))
    plt.show()


if "all_preds" in locals():
    plot_predictions(all_preds, test_ids, submission_windows)
